In [1]:
from dotenv import load_dotenv
from mem0 import Memory
import os
from openai import OpenAI
import json

In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=os.getenv("OPENAI_BASE_URL")
)

In [3]:
config = {
    "version": "v1.0",
    "embedder": {
        "provider": "openai",
        "config": {
            "api_key": OPENAI_API_KEY,
            "openai_base_url": os.getenv("OPENAI_BASE_URL"),
            "model": "text-embedding-3-small"
        }
    },
    
    "llm": {
        "provider": "openai",
        "config": {
            "api_key": OPENAI_API_KEY,
            "openai_base_url": os.getenv("OPENAI_BASE_URL"),
            "model": "gpt-4.1-mini"
        }
    },

    "vector_store": {
        "provider": "qdrant",
        "config": {
            "host": "localhost",
            "port": 6333
        }
    }
}

In [ ]:
#creating a memory client
memory_client = Memory.from_config(config)

while True:
    user_input = input("⏩")
    search_memory = memory_client.search(query=user_input, user_id="uuid")

    memories = [
        f"ID: {mem.get('id')}\nMemory: {mem.get('memory')}"
        for mem in search_memory.get("results")
    ]

    print("memories found", memories)

    SYSTEM_PROMPT = f"""
    Here is the context about the user
    {json.dumps(memories)}
    """

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        max_tokens=100,
        messages=[
            {"role": "system", "content":  SYSTEM_PROMPT},
            {"role": "user", "content":  user_input}
        ]
    )

    ai_response = response.choices[0].message.content
    print("AI Response: ", ai_response)

    memory_client.add(
        user_input, 
        user_id="uuid"
    )


    print("Memory saved successfully ✅")

memories found []
AI Response:  Hello Rohaz Bhalla! How can I assist you today?
Memory saved successfully ✅
memories found ['ID: c3c5cef8-d320-440e-8ec1-6d12e7e4d368\nMemory: Name is Rohaz Bhalla']
AI Response:  That sounds delicious! Cheese and corn make a great combo on pizza—creamy, cheesy goodness with a sweet crunch from the corn. Do you like any other toppings on your pizza?
Memory saved successfully ✅
memories found ['ID: f8d07179-f946-4a22-80c9-d7a645e04bc4\nMemory: Likes cheese and corn pizza', 'ID: c3c5cef8-d320-440e-8ec1-6d12e7e4d368\nMemory: Name is Rohaz Bhalla']
AI Response:  Since you like cheese and corn pizza, I'd recommend going for a cheese and corn pizza! It's a great combo with the creamy, melty cheese and the sweet, juicy corn kernels. If you want to try something a bit different but still along the same lines, you could also try a cheesy pizza with some other toppings like mushrooms or bell peppers to complement the corn. Would you like some recipe ideas or place